[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/03_document_structuring/03_document_structuring.ipynb)

# 03. 문서 구조화 실습 (Pydantic + OpenAI 정형 출력 + Streamlit)

> 관련 예제 프로젝트: [`example-projects/document-input-example`](https://github.com/karzit/temp/tree/master/example-projects/document-input-example) (B파트: 사용자 입력 처리) · 다른 라이브러리 선택지: [ALTERNATIVES.md](https://github.com/karzit/temp/blob/master/example-projects/document-input-example/ALTERNATIVES.md)

## 이 장을 배우는 이유

직원이 휴가신청서를 **사진으로 찍어** 올렸다고 해봅시다. [OCR](https://github.com/karzit/temp/blob/master/glossary.md#ocr)로 글자를 읽으면
이런 텍스트가 나옵니다.

```
휴가 신청서
성 명 : 김민준
신청 기간 : 2026년 8월 3일 ~ 2026년 8월 7일 (5일)
사 유 : 개인 사정으로 인한 여름 휴가 사용
2O26년 7월 2O일          ← 숫자 0을 알파벳 O로 잘못 읽음
```

사람은 읽을 수 있지만 **프로그램은 이걸로 아무것도 못 합니다.** 신청 기간을 달력에 넣으려면
`"2026-08-03"` 같은 값이 필요한데, 위 텍스트에서 그걸 꺼내려면 서식이 조금만 달라져도
깨지는 정규식을 잔뜩 짜야 합니다. 게다가 `2O26`처럼 OCR이 틀린 것도 손봐야 합니다.

**그래서 LLM에게 시킵니다.** "이 텍스트에서 서류 종류, 요청 내용, 날짜, 키워드를 뽑아줘"라고요.
LLM은 문맥을 보고 `2O26`이 2026년이라는 것도 알아챕니다.

문제는 **LLM이 자유롭게 말한다**는 것입니다. 어떨 때는 문장으로, 어떨 때는 목록으로 답합니다.
프로그램이 받아 쓰려면 **형식이 항상 같아야** 합니다. 그 형식을 강제하는 것이
[정형 출력](https://github.com/karzit/temp/blob/master/glossary.md#structured-output)(structured output)이고, 형식을 선언하는 도구가
[Pydantic](https://github.com/karzit/temp/blob/master/glossary.md#pydantic)입니다.

```
01 크롤링 → 02 청킹 → [03 구조화] → 04 RAG 검색·답변 → 05 방어
                        여기
```

이번 장에서 배우는 것

- Pydantic으로 **"데이터는 이렇게 생겨야 한다"** 를 선언하고 자동 검증하기
- **OpenAI SDK를 처음부터** — `client`가 뭔지, `messages`가 왜 목록인지, `role`이 왜 필요한지
- LLM 응답을 Pydantic 형식으로 **보장받는** 법
- [Streamlit](https://github.com/karzit/temp/blob/master/glossary.md#streamlit)으로 사용자 화면 만들기

**소요 시간**: 30~40분. **API 키가 없어도 끝까지 실행됩니다.**
키가 없으면 정규식 기반 대체 함수로 자동 전환되도록 만들어져 있습니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력이 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**, 뒤에는 **결과를 어떻게 읽는지**를 적어두었습니다.
- `structurer.py`, `app.py`처럼 나오는 파일 이름은 예제 프로젝트
  [`document-input-example`](https://github.com/karzit/temp/tree/master/example-projects/document-input-example)의 실제 파일입니다.
  **열어보지 않아도 따라갈 수 있습니다.**
- 낯선 용어는 [glossary.md](https://github.com/karzit/temp/blob/master/glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| 결과가 투박하고 `keywords`가 부실하다 | **API 키가 없어 규칙 기반 대체 함수로 돌았습니다** | 정상이고, 의도된 동작입니다. LLM과 규칙의 차이를 보는 것이 실습 4의 목적입니다 |
| `KeyError: 'OPENAI_API_KEY'` | `.env`에 키를 넣었지만 `load_dotenv()`보다 먼저 호출됨 | 위 셀부터 순서대로 실행. 값을 방금 고쳤다면 `load_dotenv(override=True)` |
| `AttributeError`: `.beta...parse`가 없다 | `openai` SDK 버전이 낮음 (정형 출력은 1.40 이상 필요) | `pip install -U openai` 후 런타임 재시작 |
| `BadRequestError` — `response_format` 관련 | 쓰는 모델이 **정형 출력을 지원하지 않음** | `gpt-4o-mini` 이상을 쓰세요. 구형 모델은 Pydantic 스키마를 못 받습니다 |
| `RateLimitError` / `insufficient_quota` | 키는 유효하지만 크레딧이 없거나 호출이 몰림 | 결제 상태 확인. 급하지 않다면 키를 지우고 규칙 기반으로 진행해도 됩니다 |
| `ValidationError` | Pydantic이 타입 불일치를 잡은 것 | **실습 1에서는 일부러 냅니다.** 그 외 자리에서 나면 LLM이 형식을 틀린 것이니, 그걸 잡아내는 게 이 장의 요지입니다 |

여기 없는 문제(설치 실패, 한글 깨짐, API 키 설정 방법)는 [troubleshooting.md](https://github.com/karzit/temp/blob/master/troubleshooting.md)에 모아뒀습니다.

## 실습 0. 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pydantic openai python-dotenv

## 실습 1. Pydantic 모델 정의하기

예제 프로젝트 `document-input-example`의 `structurer.py`에 있는 `RegulationInquiry`를
그대로 만들어봅니다.

`Field(description=...)`는 사람이 읽는 설명이기만 한 게 아닙니다.
OpenAI에게 "이 필드에는 이런 내용을 채워라"라고 알려주는 힌트로도 쓰입니다.

In [ ]:
# Pydantic: '이 데이터는 이런 형태여야 한다'를 클래스로 선언하고 자동 검증해주는 라이브러리
from pydantic import BaseModel, Field, ValidationError


class RegulationInquiry(BaseModel):
    document_type: str = Field(description="서류의 종류 (예: 휴가신청서, 재직증명서, 초과근무신청서 등)")
    applicant_request: str = Field(description="신청자가 요청하는 핵심 내용을 한두 문장으로 요약")
    related_dates: list[str] = Field(
        default_factory=list, description="서류에 등장하는 날짜들 (YYYY-MM-DD 형식)"
    )
    keywords: list[str] = Field(
        default_factory=list, description="규정 검색에 도움이 될 핵심 키워드 목록"
    )


print(RegulationInquiry.model_json_schema())

**결과 읽는 법** — 우리가 파이썬 클래스로 쓴 것이 **JSON 스키마**라는 형식으로 출력됩니다.
`document_type`은 문자열, `related_dates`는 문자열의 배열... 이런 설명서입니다.

**이 설명서가 그대로 OpenAI에게 전달됩니다.** 그래서 `Field(description=...)`에 적은 말이
중요합니다. 사람이 읽는 주석이 아니라 **LLM에게 주는 지시**이기 때문입니다.
`"날짜들 (YYYY-MM-DD 형식)"`이라고 써두면 LLM이 그 형식으로 맞춰서 채웁니다.

`list[str]`처럼 파이썬 타입만 적으면 [Pydantic](https://github.com/karzit/temp/blob/master/glossary.md#pydantic)이 알아서 스키마로 바꿔줍니다.
**우리는 파이썬만 쓰고, 나머지는 라이브러리가 합니다.**


Pydantic은 타입이 맞지 않는 값을 넣으면 즉시 에러를 내서 잘못된 데이터가 조용히 통과하는 것을 막아줍니다.

In [ ]:
try:
    RegulationInquiry(document_type=123, applicant_request="휴가 신청", related_dates="2026-01-01")
except ValidationError as e:
    print("검증 실패:")
    print(e)

**결과 읽는 법** — 일부러 틀린 값을 넣어 에러를 내봤습니다.

- `document_type=123` → 문자열이어야 하는데 숫자를 넣었습니다
- `related_dates="2026-01-01"` → 목록이어야 하는데 문자열 하나를 넣었습니다

Pydantic이 **어느 필드가 왜 틀렸는지** 조목조목 알려줍니다.

**이게 왜 중요할까요?** 검증이 없으면 잘못된 데이터가 **조용히 통과해서** 훨씬 나중에,
전혀 상관없어 보이는 곳에서 터집니다. "날짜 목록을 순회하려는데 왜 글자 하나씩 나오지?" 같은
버그를 몇 시간씩 쫓게 됩니다.

**LLM은 가끔 형식을 틀립니다.** 그 순간을 여기서 잡아내는 것입니다.


## 실습 2. OCR 원문처럼 지저분한 텍스트 준비하기

실제 프로젝트에서는 `google-cloud-vision`이 서류 사진에서 이 텍스트를 뽑아냅니다.
Google Cloud 인증이 필요해서 이 노트북에서는 다루지 않고,
**OCR 결과를 흉내 낸 예시 텍스트**를 준비합니다.

In [ ]:
raw_ocr_text = (
    "휴가 신청서\n"
    "성 명 : 김민준\n"
    "부 서 : 개발팀\n"
    "신청 기간 : 2026년 8월 3일 ~ 2026년 8월 7일 (5일)\n"
    "사 유 : 개인 사정으로 인한 여름 휴가 사용\n"
    "위와 같이 연차휴가를 신청합니다.\n"
    "2O26년 7월 2O일\n"  # OCR 특유의 오타: 숫자 0이 알파벳 O로 잘못 인식됨
    "신청인 : 김민준 (인)"
)
print(raw_ocr_text)

**결과 읽는 법** — `2O26년 7월 2O일`을 자세히 보세요. **숫자 0이 아니라 알파벳 대문자 O**입니다.
OCR이 가장 흔하게 틀리는 유형입니다(0↔O, 1↔l↔I, 5↔S).

이런 텍스트에서 날짜를 정규식으로 뽑으려면 `[0-9oO]` 같은 패턴을 써야 하고,
서식이 조금만 달라지면 또 깨집니다. **LLM은 문맥을 보고 "2026년이겠구나"를 알아냅니다.**
이 차이가 다음 실습의 요점입니다.


## 실습 3. OpenAI로 정형 데이터 만들기 (키가 없으면 규칙 기반 대체 함수 사용)

`OPENAI_API_KEY` 환경변수가 설정되어 있으면 실제 `gpt-4o-mini` 호출로 파싱합니다.
없으면 아주 단순한 규칙 기반(정규식·키워드) 함수로 대체합니다.
덕분에 **API 키나 비용 없이도** 실습을 끝까지 진행할 수 있습니다.

> **아래 셀은 네 부분입니다.** 코드가 길지만 구조는 단순합니다.
> 1. `SYSTEM_PROMPT` — LLM에게 "무엇을 뽑아달라"고 지시하는 문장
> 2. `structure_with_openai()` — **키가 있을 때** OpenAI로 구조화
> 3. `structure_with_rules()` — **키가 없을 때** 정규식으로 흉내 내는 대체 함수
> 4. `structure_text()` — 위 둘 중 하나를 자동으로 골라 부르는 입구
>
> 앞으로는 `structure_text()`만 부르면 되고, 키 유무는 이 함수가 알아서 처리합니다.

### 잠깐, OpenAI에 요청을 보내는 코드를 먼저 읽어봅시다

다음 셀에 이런 코드가 나옵니다. **처음 보면 하나도 안 읽히는 게 정상입니다.** 한 줄씩 뜯어봅니다.

```python
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": raw_text},
    ],
    response_format=RegulationInquiry,
)
return completion.choices[0].message.parsed
```

**① `client = OpenAI(...)` — 왜 객체를 만드나요?**

OpenAI는 우리 컴퓨터가 아니라 **인터넷 저편의 서버**에 있습니다. 거기에 요청을 보내려면
주소·인증키·재시도 규칙 같은 걸 알고 있는 **"통신 담당자"** 가 필요합니다.
`OpenAI(...)`가 그 담당자를 만드는 코드이고, 관례적으로 `client`라는 이름을 붙입니다.
한 번 만들어두면 그 담당자에게 계속 심부름을 시킬 수 있습니다.

**② `client.beta.chat.completions.parse(...)` — 왜 이렇게 기나요?**

점(`.`)으로 이어진 것은 **분류 서랍**이라고 보면 됩니다.
`client` 안에 `chat`(대화 기능) 서랍이 있고, 그 안에 `completions`(답변 생성) 서랍이 있고,
거기서 `parse`(정해진 형식으로 파싱해서 받기)를 꺼내 쓰는 것입니다.
그냥 문장으로 답을 받고 싶으면 `client.chat.completions.create(...)`를 씁니다.
(`beta`는 아직 실험 단계인 기능이라는 표시입니다.)

**③ `messages`는 왜 목록(리스트)인가요?**

대화는 한 번의 질문으로 끝나지 않기 때문입니다. LLM에게는 **대화 기록 전체**를 매번 통째로
보내야 합니다. **LLM은 지난 대화를 기억하지 못합니다.** 우리가 매번 다시 알려주는 것입니다.

**④ `role`은 왜 필요한가요?**

같은 글이라도 **누가 한 말인지**에 따라 무게가 다르기 때문입니다.

| role | 누구의 말 | 쓰임 |
|---|---|---|
| `system` | 개발자가 정한 규칙 | "너는 이런 역할이다", "이건 하지 마라" |
| `user` | 사용자 | 실제 질문이나 입력 |
| `assistant` | AI가 이전에 한 답 | 대화를 이어갈 때 |

여기서는 `system`에 "지어내지 말라"는 지시를 넣고, `user`에 OCR 텍스트를 넣었습니다.
**이 구분이 05번([프롬프트 인젝션](https://github.com/karzit/temp/blob/master/glossary.md#prompt-injection) 방어)의 핵심 무기가 됩니다.** 문서 내용을 `system`에 넣으면
문서 안에 숨겨진 지시문이 규칙 행세를 할 수 있기 때문입니다.

**⑤ `completion.choices[0].message.parsed` — 왜 이렇게 파고들어야 하나요?**

응답이 여러 겹으로 포장되어 오기 때문입니다.

```
completion            ← 응답 전체 (사용 토큰 수, 모델 이름 등도 들어 있음)
  .choices            ← 답변 후보 목록 (여러 개를 요청할 수도 있어서 목록)
  [0]                 ← 그중 첫 번째
  .message            ← 그 답변의 메시지 부분
  .parsed             ← Pydantic 객체로 변환된 결과
```

보통 답변을 하나만 요청하므로 `choices[0]`이 관용구처럼 굳어져 있습니다.
`.parse` 대신 `.create`를 썼다면 마지막이 `.parsed`가 아니라 `.content`(그냥 문자열)입니다.


In [ ]:
import os
import re
from dotenv import load_dotenv   # load_dotenv: .env 파일의 값을 환경 변수로 읽어들인다(API 키 보관용)

load_dotenv()

# ══════════════════════════════════════════════════════════════
# ① 시스템 프롬프트 — LLM에게 '무엇을, 어떤 태도로' 뽑을지 지시
#    핵심은 마지막 문장: 없는 내용을 지어내지 말라(환각 방지)
# ══════════════════════════════════════════════════════════════
SYSTEM_PROMPT = (
    "당신은 사내 서류 텍스트를 분석해 정해진 형식의 데이터로 정리하는 도우미입니다. "
    "OCR로 인식된 텍스트라 오타나 줄바꿈 오류가 있을 수 있으니, 문맥을 보고 자연스럽게 해석해 채우세요. "
    "텍스트에 없는 내용은 추측해서 지어내지 말고, 알 수 없으면 빈 값으로 두세요."
)


# ══════════════════════════════════════════════════════════════
# ② API 키가 있을 때 — OpenAI에게 구조화를 맡긴다
# ══════════════════════════════════════════════════════════════
def structure_with_openai(raw_text: str) -> RegulationInquiry:
    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": raw_text},
        ],
        # response_format에 Pydantic 모델을 주면, 응답이 그 형식에 맞는지
        # OpenAI 쪽에서 보장해준다. 파싱 실패를 걱정하지 않아도 된다.
        response_format=RegulationInquiry,
    )
    return completion.choices[0].message.parsed


# ══════════════════════════════════════════════════════════════
# ③ API 키가 없을 때 — 정규식으로 흉내 내는 대체 함수
#    LLM 없이도 노트북이 끝까지 돌아가게 하려는 장치다
# ══════════════════════════════════════════════════════════════
def structure_with_rules(raw_text: str) -> RegulationInquiry:
    """API 키 없이 실습할 수 있도록 만든 아주 단순한 규칙 기반 대체 함수."""
    doc_type = "휴가신청서" if "휴가" in raw_text else "미분류 문서"

    # '사 유 :' 뒤의 문장을 통째로 잡는다. \s*는 띄어쓰기가 몇 개든 허용한다는 뜻
    reason_match = re.search(r"사\s*유\s*:\s*(.+)", raw_text)
    reason = reason_match.group(1).strip() if reason_match else ""

    # '2O26', '2O일'처럼 OCR이 숫자 0을 O로 잘못 읽은 경우까지 연/월/일 각 자리에서 느슨하게 잡아낸다
    date_pattern = r"([0-9oO]{4})[년.\-]\s*([0-9oO]{1,2})[월.\-]\s*([0-9oO]{1,2})"

    def fix_ocr_digits(s: str) -> str:
        return s.replace("O", "0").replace("o", "0")

    dates = []
    for y, m, d in re.findall(date_pattern, raw_text):
        y, m, d = fix_ocr_digits(y), fix_ocr_digits(m), fix_ocr_digits(d)
        # OCR 오독을 보정해도 말이 안 되는 날짜(0000년, 13월, 32일 등)가 나올 수 있다.
        # 검증 없이 넣으면 존재하지 않는 날짜가 실제 날짜인 것처럼 결과에 섞여 들어간다.
        if not (1900 <= int(y) <= 2100 and 1 <= int(m) <= 12 and 1 <= int(d) <= 31):
            continue
        dates.append(f"{y}-{int(m):02d}-{int(d):02d}")

    # 미리 정해둔 목록에 있는 단어만 키워드로 인정 (LLM처럼 새 단어를 만들어내지 못한다)
    keywords = [kw for kw in ["연차휴가", "육아휴직", "재택근무", "경조휴가"] if kw in raw_text]
    if not keywords and doc_type == "휴가신청서":
        keywords = ["연차휴가"]

    return RegulationInquiry(
        document_type=doc_type, applicant_request=reason, related_dates=dates, keywords=keywords
    )


# ══════════════════════════════════════════════════════════════
# ④ 입구 함수 — 앞으로는 이것만 부르면 된다
#    키가 있으면 ②, 없으면 ③으로 자동 분기
# ══════════════════════════════════════════════════════════════
def structure_text(raw_text: str) -> RegulationInquiry:
    if os.getenv("OPENAI_API_KEY"):
        return structure_with_openai(raw_text)
    print("(OPENAI_API_KEY가 없어 규칙 기반 대체 함수로 실행합니다)")
    return structure_with_rules(raw_text)

## 실습 4. 결과 확인하기

In [ ]:
structured = structure_text(raw_ocr_text)
print(structured.model_dump_json(indent=2))

**결과 읽는 법** — 지저분한 OCR 텍스트가 **프로그램이 바로 쓸 수 있는 JSON**이 되었습니다.

```
"document_type": "휴가신청서"
"related_dates": ["2026-08-03", "2026-08-07", "2026-07-20"]
"keywords": ["연차휴가", ...]
```

`related_dates`가 `YYYY-MM-DD` 형식으로 정리된 것을 보세요. 이제 이 값을 그대로
달력에 넣거나 DB에 저장할 수 있습니다.

**API 키가 없어 규칙 기반으로 돌았다면** 결과가 조금 투박할 것입니다.
`keywords`는 미리 정해둔 목록에서만 고를 수 있어서, 새로운 표현이 나오면 놓칩니다.
**LLM과 규칙의 차이가 바로 여기입니다.** 규칙은 예상한 것만 처리하고,
LLM은 처음 보는 서식도 문맥으로 처리합니다. 대신 LLM은 **가끔 틀립니다.**

`keywords`가 왜 필요한지도 짚고 갑니다. 다음 04번에서 **이 키워드로 규정을 검색**합니다.
사용자가 올린 서류에서 "연차휴가"를 뽑아내면, 그 단어로 규정집을 뒤져 관련 조항을 찾는 식입니다.

**규칙 기반 함수에는 또 다른 위험이 있습니다.** OCR 오타 보정(`O`→`0`)이 지나치면
"OOOO년 O월 O일"처럼 아예 읽을 수 없는 텍스트도 `"0000-00-00"`이라는 그럴듯한 형식의
가짜 날짜로 둔갑시킬 수 있습니다. LLM과 달리 규칙 기반 함수는 스스로 "이건 말이 안 되는데?"라고
판단하지 못하기 때문에, `structure_with_rules()`는 연/월/일 범위를 벗어난 값을
**아예 결과에서 제외**하도록 만들어져 있습니다. 정규식으로 무언가를 "느슨하게" 잡아낼 때는
잡아낸 값이 실제로 말이 되는지 검증하는 단계를 항상 같이 둬야 합니다.


## 실습 5. Streamlit 앱 형태 살펴보기

`app.py`는 `st.file_uploader()`로 이미지를 받아 `structure_text()`의 결과를 화면에
보여주는 구조입니다. 노트북에서 직접 띄우는 대신 파일로 저장해두고,
실제로는 터미널에서 `streamlit run app.py`로 실행합니다.

In [ ]:
app_code = '''
import streamlit as st

st.set_page_config(page_title="서류 사진 -> 정형 데이터 변환", page_icon="\U0001F4C4")
st.title("\U0001F4C4 서류 사진 -> 정형 데이터 변환")

uploaded_file = st.file_uploader("서류 이미지를 업로드하세요", type=["png", "jpg", "jpeg"])

if uploaded_file is not None:
    st.image(uploaded_file, caption="업로드한 이미지", use_container_width=True)
    if st.button("분석 시작"):
        # 실제로는 OCR(extract_text_from_image) -> structure_text() 순서로 처리합니다.
        st.write("여기에서 OCR과 정형화 결과를 보여줍니다.")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py 저장 완료. 로컬에서는 다음 명령으로 실행합니다:")
print("  streamlit run app.py")

## 정리

이번 장에서 한 일

1. **Pydantic**으로 받아야 할 데이터의 모양을 선언하고, 타입이 틀리면 즉시 걸러냈습니다
2. **OpenAI SDK**의 `client` / `messages` / `role` / `choices[0]`이 각각 무엇인지 익혔습니다
3. `response_format`에 Pydantic 모델을 넘겨 **응답 형식을 보장**받았습니다
4. API 키가 없을 때를 대비한 **대체 경로**를 두는 설계를 봤습니다
5. Streamlit으로 사용자 화면의 뼈대를 만들었습니다

**가장 기억할 것**: **LLM의 출력을 그대로 믿고 프로그램에 넣으면 안 됩니다.**
형식을 선언하고(Pydantic), 그 형식을 강제하고(`response_format`), 그래도 틀리면 걸러냅니다.
LLM은 확률적으로 답하는 도구라 **가끔은 반드시 이상하게 답합니다.**

**스스로 확인해보기**

- [ ] `client = OpenAI(...)`가 무엇을 만드는 코드인지 말할 수 있다
- [ ] `messages`가 왜 목록인지, `role`이 왜 필요한지 설명할 수 있다
- [ ] `choices[0]`을 왜 거쳐야 하는지 안다
- [ ] `system`과 `user`에 무엇을 넣어야 하는지 구분할 수 있다
- [ ] 정형 출력이 없으면 무엇이 문제인지 예를 들 수 있다

## 연습 문제

1. `RegulationInquiry`에 `urgency`(긴급도: `"높음"`/`"보통"`/`"낮음"`) 필드를 추가하고,
   `structure_with_rules()`에도 규칙(예: "긴급", "즉시"가 있으면 `"높음"`)을 넣어 반영해보세요.
2. 서로 다른 서류 텍스트 3개를 준비하고, `structure_text()`를 배치로 호출해
   `RegulationInquiry` 목록을 만드는 함수를 작성해보세요.
3. `SYSTEM_PROMPT`에서 "지어내지 말라"는 문장을 빼고 실행하면 결과가 달라지나요?
   (API 키가 있을 때만 확인할 수 있습니다.)

**해설/정답**: [03_document_structuring_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/03_document_structuring/03_document_structuring_solutions.ipynb)

## 다음 단계

다음 노트북([04_rag_pipeline](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/04_rag_pipeline/04_rag_pipeline.ipynb))에서 드디어
**[RAG](https://github.com/karzit/temp/blob/master/glossary.md#rag) 본체**를 만듭니다. 02번에서 잘라둔 조각들 중 질문과 관련 있는 것을 찾아내
LLM에게 함께 넘기는 전 과정을 직접 구현합니다.
